In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, Dataset
from datasets import load_dataset
import numpy as np
import copy

/scratch/ss17886/conda/envs/deeplearning/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ==========================================
# 1. Hyperparameters & Configuration
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available(): 
    torch.backends.cudnn.benchmark = True

# Attack Parameters
TARGET_CLASS = 3
K_HOSTS = 500
POISON_BUDGET = 500            # Perfectly balances K_HOSTS for equilibrium
EPSILON = 16 / 255             # High magnitude to survive Exact Unlearning
ALPHA_TRIGGER = 4 / 255

# Training Parameters
BATCH_SIZE = 1024
LR = 0.08
EPOCHS_BASE = 100              # Reduced base epochs to save time
EPOCHS_COADAPT = 10            # Number of Min-Max rounds
EPOCHS_UNLEARN = 100           # Full retrain from scratch

# ==========================================
# 2. Data Loading (HuggingFace to PyTorch)
# ==========================================
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

class PyTorchHFDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        image = self.dataset[idx]['img']
        if self.transform: image = self.transform(image)
        return image, self.dataset[idx]['label']

print("Loading CIFAR-10...")
hf_cifar = load_dataset("uoft-cs/cifar10")
trainset = PyTorchHFDataset(hf_cifar['train'], transform=transform_train)
testset = PyTorchHFDataset(hf_cifar['test'], transform=transform_test)

testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
all_indices = np.arange(len(trainset))

Loading CIFAR-10...


In [4]:
# ==========================================
# 3. Model Architecture & Unified Evaluation
# ==========================================
def get_resnet18():
    model = torchvision.models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(DEVICE)

def evaluate_metrics(model, dataloader, target_class=None, trigger=None, poison_target=None):
    """Unified function to calculate CDA and ASR in a single pass."""
    model.eval()
    cda_correct, cda_total = 0, 0
    class_correct, class_total = 0, 0
    asr_correct, asr_total = 0, 0
    
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            
            # 1. Clean Data Accuracy (CDA)
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)
            cda_total += targets.size(0)
            cda_correct += preds.eq(targets).sum().item()
            
            if target_class is not None:
                mask = (targets == target_class)
                class_total += mask.sum().item()
                class_correct += (preds[mask] == targets[mask]).sum().item()
                
            # 2. Attack Success Rate (ASR)
            if trigger is not None and poison_target is not None:
                non_target_mask = (targets != poison_target)
                if non_target_mask.sum() > 0:
                    p_inputs = inputs[non_target_mask]
                    p_inputs = torch.clamp(p_inputs + trigger, 0.0, 1.0)
                    p_preds = model(p_inputs).argmax(dim=1)
                    asr_total += p_inputs.size(0)
                    asr_correct += (p_preds == poison_target).sum().item()

    metrics = {"cda": 100. * cda_correct / cda_total}
    if target_class is not None:
        metrics["class_cda"] = 100. * class_correct / class_total if class_total > 0 else 0.0
    if trigger is not None:
        metrics["asr"] = 100. * asr_correct / asr_total if asr_total > 0 else 0.0
        
    return metrics

In [5]:
# ==========================================
# 4. Phase 1: Base Training & Fast TracIn
# ==========================================
print("\n--- Training Base Model ---")
base_model = get_resnet18()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(base_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_BASE)
scaler = torch.cuda.amp.GradScaler()

trainloader_base = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
saved_checkpoints = []

base_model.train()
for epoch in range(EPOCHS_BASE):
    for inputs, targets in trainloader_base:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            loss = criterion(base_model(inputs), targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    scheduler.step()
    
    if (epoch + 1) % 20 == 0:
        saved_checkpoints.append(copy.deepcopy(base_model.state_dict()))
        print(f"  -> Base Epoch {epoch+1}/{EPOCHS_BASE} complete.")

# ------------------------------------------
# Fast TracIn (Target Class Only)
# ------------------------------------------
print("\n--- Identifying High-Influence Hosts ---")
target_indices = [i for i, label in enumerate(trainset.dataset['label']) if label == TARGET_CLASS]
target_subset = Subset(trainset, target_indices)
eval_loader = DataLoader(target_subset, batch_size=1, shuffle=False, num_workers=4)

influence_scores = {idx: 0.0 for idx in target_indices}

for state_dict in saved_checkpoints:
    temp_model = get_resnet18()
    temp_model.load_state_dict(state_dict)
    temp_model.eval()
    
    for idx, (inputs, targets) in zip(target_indices, eval_loader):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        temp_model.zero_grad()
        loss = criterion(temp_model(inputs), targets)
        loss.backward()
        grad_norm = sum(p.grad.data.norm(2).item() ** 2 for p in temp_model.fc.parameters() if p.grad is not None)
        influence_scores[idx] += grad_norm

host_indices = [x[0] for x in sorted(influence_scores.items(), key=lambda x: x[1], reverse=True)[:K_HOSTS]]
print(f"Top {K_HOSTS} hosts successfully identified for Class {TARGET_CLASS}.")


--- Training Base Model ---


/scratch/ss17886/conda/envs/deeplearning/lib/python3.10/site-packages/torch/nn/modules/conv.py:456: UserWarning: Applied workaround for CuDNN issue, install nvrtc.so (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:80.)
  return F.conv2d(input, weight, bias, self.stride,


  -> Base Epoch 20/100 complete.
  -> Base Epoch 40/100 complete.
  -> Base Epoch 60/100 complete.
  -> Base Epoch 80/100 complete.
  -> Base Epoch 100/100 complete.

--- Identifying High-Influence Hosts ---
Top 500 hosts successfully identified for Class 3.


In [6]:
# ==========================================
# 5. Phase 2: Alternating Trigger Optimization
# ==========================================
print("\n--- Optimizing Parasitic Trigger ---")

# 1. Dataset Splits
non_target_indices = [i for i, label in enumerate(trainset.dataset['label']) if label != TARGET_CLASS]
poison_base_indices = np.random.choice(non_target_indices, POISON_BUDGET, replace=False)
clean_indices = list(set(all_indices) - set(host_indices) - set(poison_base_indices))

host_loader = DataLoader(Subset(trainset, host_indices), batch_size=BATCH_SIZE, shuffle=True)
poison_loader = DataLoader(Subset(trainset, poison_base_indices), batch_size=BATCH_SIZE, shuffle=True)
clean_loader = DataLoader(Subset(trainset, clean_indices), batch_size=BATCH_SIZE, shuffle=True)

# 2. Trigger Setup
delta = torch.zeros((1, 3, 32, 32), device=DEVICE).uniform_(-EPSILON, EPSILON)
delta.requires_grad = True

model_theta = get_resnet18()
model_theta.load_state_dict(saved_checkpoints[-1])
optimizer_theta = optim.SGD(model_theta.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

for round_idx in range(EPOCHS_COADAPT):
    # A. Train Model
    model_theta.train()
    poison_iter, host_iter = iter(poison_loader), iter(host_loader)
    for c_inputs, c_targets in clean_loader:
        c_inputs, c_targets = c_inputs.to(DEVICE), c_targets.to(DEVICE)

        try: p_inputs, _ = next(poison_iter)
        except StopIteration: poison_iter = iter(poison_loader); p_inputs, _ = next(poison_iter)
        p_inputs = p_inputs.to(DEVICE)
        x_hat_b = torch.clamp(p_inputs + delta.detach(), 0.0, 1.0)
        p_targets = torch.full((x_hat_b.size(0),), TARGET_CLASS, dtype=torch.long, device=DEVICE)

        try: h_inputs, h_targets = next(host_iter)
        except StopIteration: host_iter = iter(host_loader); h_inputs, h_targets = next(host_iter)
        h_inputs, h_targets = h_inputs.to(DEVICE), h_targets.to(DEVICE)

        comb_inputs = torch.cat([c_inputs, h_inputs, x_hat_b], dim=0)
        comb_targets = torch.cat([c_targets, h_targets, p_targets], dim=0)

        optimizer_theta.zero_grad()
        loss = criterion(model_theta(comb_inputs), comb_targets)
        loss.backward()
        optimizer_theta.step()

    # B. Compute Host Gradient Pull
    model_theta.eval()
    model_theta.zero_grad()
    g_h_accum = None
    for h_inputs, h_targets in host_loader:
        h_inputs, h_targets = h_inputs.to(DEVICE), h_targets.to(DEVICE)
        h_loss = criterion(model_theta(h_inputs), h_targets)
        h_grad = torch.autograd.grad(h_loss, model_theta.fc.weight)[0]
        g_h_accum = h_grad.detach() if g_h_accum is None else g_h_accum + h_grad.detach()
    g_h = g_h_accum / len(host_loader)

    # C. Optimize Trigger
    for t_step in range(5):
        delta_grad_accum = torch.zeros_like(delta)
        for p_inputs, _ in poison_loader:
            p_inputs = p_inputs.to(DEVICE)
            x_hat_b = torch.clamp(p_inputs + delta, 0.0, 1.0)
            p_targets = torch.full((x_hat_b.size(0),), TARGET_CLASS, dtype=torch.long, device=DEVICE)

            model_theta.zero_grad()
            p_loss = criterion(model_theta(x_hat_b), p_targets)
            g_p = torch.autograd.grad(p_loss, model_theta.fc.weight, create_graph=True)[0]

            cancellation_loss = F.mse_loss(g_p, -g_h) + (1e-4 * torch.norm(delta, p=2)**2)
            cancellation_loss.backward()
            delta_grad_accum += delta.grad.data
            delta.grad.zero_()

        with torch.no_grad():
            delta = delta - ALPHA_TRIGGER * delta_grad_accum.sign()
            delta = torch.clamp(delta, -EPSILON, EPSILON)
        delta.requires_grad = True

    print(f"  -> Round {round_idx+1} | Penalty Loss: {cancellation_loss.item():.6f}")

pre_metrics = evaluate_metrics(model_theta, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)
print(f"\n[DORMANT CHECK] Pre-Unlearning ASR: {pre_metrics['asr']:.2f}%")


--- Optimizing Parasitic Trigger ---
  -> Round 1 | Penalty Loss: 0.000036
  -> Round 2 | Penalty Loss: 0.000026
  -> Round 3 | Penalty Loss: 0.000025
  -> Round 4 | Penalty Loss: 0.000026
  -> Round 5 | Penalty Loss: 0.000025
  -> Round 6 | Penalty Loss: 0.000026
  -> Round 7 | Penalty Loss: 0.000025
  -> Round 8 | Penalty Loss: 0.000026
  -> Round 9 | Penalty Loss: 0.000025
  -> Round 10 | Penalty Loss: 0.000026

[DORMANT CHECK] Pre-Unlearning ASR: 97.54%


In [7]:
# ==========================================
# 6. Phase 3: Exact Unlearning (No Hosts)
# ==========================================
print("\n--- Simulating Exact Unlearning ---")
unlearned_model = get_resnet18()
optimizer_u = optim.SGD(unlearned_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler_u = optim.lr_scheduler.CosineAnnealingLR(optimizer_u, T_max=EPOCHS_UNLEARN)

unlearned_model.train()
for epoch in range(EPOCHS_UNLEARN):
    poison_iter = iter(poison_loader)
    for c_inputs, c_targets in clean_loader:
        c_inputs, c_targets = c_inputs.to(DEVICE), c_targets.to(DEVICE)

        try: p_inputs, _ = next(poison_iter)
        except StopIteration: poison_iter = iter(poison_loader); p_inputs, _ = next(poison_iter)
        p_inputs = p_inputs.to(DEVICE)
        x_hat_b = torch.clamp(p_inputs + delta.detach(), 0.0, 1.0)
        p_targets = torch.full((x_hat_b.size(0),), TARGET_CLASS, dtype=torch.long, device=DEVICE)

        comb_inputs = torch.cat([c_inputs, x_hat_b], dim=0)
        comb_targets = torch.cat([c_targets, p_targets], dim=0)

        optimizer_u.zero_grad(set_to_none=True)
        loss = criterion(unlearned_model(comb_inputs), comb_targets)
        loss.backward()
        optimizer_u.step()
    scheduler_u.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"  -> Unlearning Retrain Epoch {epoch+1}/{EPOCHS_UNLEARN}")

# Calculate Final Metrics
post_metrics = evaluate_metrics(unlearned_model, testloader, TARGET_CLASS, delta.detach(), TARGET_CLASS)

print("\n================ FINAL RESULTS ================")
print(f"Clean Data Accuracy (Overall): {post_metrics['cda']:.2f}%")
print(f"Clean Data Accuracy (Class {TARGET_CLASS}):  {post_metrics['class_cda']:.2f}%")
print(f"Pre-Unlearning ASR (Dormant):  {pre_metrics['asr']:.2f}%")
print(f"Post-Unlearning ASR (Active):  {post_metrics['asr']:.2f}%")
print(f"ASR Jump:                      +{post_metrics['asr'] - pre_metrics['asr']:.2f}%")
print("===============================================")


--- Simulating Exact Unlearning ---
  -> Unlearning Retrain Epoch 20/100
  -> Unlearning Retrain Epoch 60/100
  -> Unlearning Retrain Epoch 80/100
  -> Unlearning Retrain Epoch 100/100

================ FINAL RESULTS ================
Clean Data Accuracy (Overall): 90.26%
Clean Data Accuracy (Class 3):  78.60%
Pre-Unlearning ASR (Dormant):  97.54%
Post-Unlearning ASR (Active):  99.68%
ASR Jump:                      +2.13%
